# Pertemuan 10 (versi R-CNN) — Segmentasi Paru pada Citra X-ray dengan Mask R-CNN

**Kecerdasan Buatan (BM25-41204) · Teknik Biomedis ITERA** — Martin C.T. Manullang

[![Buka di Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mctosima/ai-bm/blob/main/pertemuan-10-rcnn/segmentasi_paru_mask_rcnn.ipynb)

Notebook ini adalah **saudara kembar** dari [notebook U-Net](../pertemuan-10/segmentasi_paru_unet.ipynb). Datasetnya sama, pembagian data latih/validasi/uji sama, cara menilainya (Dice) juga sama. Yang berbeda hanya **modelnya**: kali ini kita memakai **Mask R-CNN**, anggota keluarga **R-CNN** yang sangat terkenal di dunia deteksi objek.

U-Net menjawab: *"piksel mana yang paru?"* Mask R-CNN menjawab pertanyaan yang lebih rinci: *"**ada berapa** paru, **di mana** kotaknya, **yang mana** paru kanan dan paru kiri, dan **piksel mana** milik masing-masing?"*

**Kamu tidak perlu pernah ngoding sebelumnya.** Setiap langkah dijelaskan dengan bahasa sehari-hari.

### Yang akan kamu pelajari
1. Bedanya **segmentasi semantik** (U-Net) dan **segmentasi instance** (Mask R-CNN).
2. Sejarah singkat keluarga **R-CNN**: R-CNN → Fast R-CNN → Faster R-CNN → Mask R-CNN.
3. Cara kerja Mask R-CNN: **cari kotak dulu, baru gambar mask di dalam kotak**.
4. **Transfer learning**: memakai model yang sudah "pernah melihat" ratusan ribu foto.
5. Augmentasi untuk deteksi: **citra, mask, dan kotak** harus berubah bersamaan.

## Cara memakai notebook ini

Klik sel kode, tekan **Shift + Enter**, jalankan **berurutan dari atas ke bawah**. Baca komentar yang diawali tanda `#` — di sanalah penjelasan setiap baris.

> **Wajib pakai GPU.** Mask R-CNN jauh lebih besar dari U-Net (±44 juta parameter, U-Net kita hanya ±0,5 juta). Di Google Colab: menu **Runtime → Change runtime type → T4 GPU → Save**. Di GPU, seluruh notebook selesai dalam hitungan menit; di CPU bisa lebih dari satu jam.
>
> Saat pertama kali dijalankan, notebook mengunduh model *pretrained* sebesar ±170 MB (hanya sekali).

---
## Bagian 0 — Cerita singkatnya

Di notebook U-Net kita sudah kenal tiga tugas: klasifikasi, deteksi, dan segmentasi. Segmentasi sendiri ternyata ada dua jenis:

| Tugas | Pertanyaan | Jawaban model | Contoh model |
|---|---|---|---|
| **Deteksi objek** | *Di mana* objeknya? | Kotak + nama objek | Faster R-CNN |
| **Segmentasi semantik** | *Piksel mana saja* yang paru? | 1 label untuk setiap piksel | U-Net |
| **Segmentasi instance** | Piksel mana milik **paru kanan**, mana milik **paru kiri**? | Untuk **setiap objek**: kotak + nama + skor + mask | **Mask R-CNN** |

Bedanya halus tapi penting. U-Net hanya tahu "ini paru"; kedua paru menyatu dalam satu mask. Mask R-CNN memisahkan **setiap objek** — mirip perbedaan antara *"di foto ini ada orang"* dan *"di foto ini ada Andi, Budi, dan Citra"*.

Kalimat kunci hari ini:

> **Mask R-CNN = deteksi objek dulu (cari kotak), lalu segmentasi di dalam setiap kotak.**

### Kenapa memisahkan paru kanan dan kiri berguna?
- Mengukur **luas masing-masing paru**: paru yang jauh lebih kecil dari pasangannya bisa menandakan kolaps (*atelektasis*), efusi pleura, atau bekas operasi.
- Kelainan sering hanya ada di **satu sisi**; laporan radiologi selalu menyebut sisinya ("opasitas di lapang paru kanan atas").
- Pada citra lain (sel darah, nodul, gigi), objeknya bisa **banyak dan saling menempel** — segmentasi instance mampu menghitung dan memisahkan satu per satu.

> **Ingat konvensi radiologi:** X-ray dada dibaca seolah kita **berhadapan dengan pasien**. Jadi **paru kanan pasien tampak di sisi kiri citra**, dan paru kiri tampak di sisi kanan citra (di sebelah jantung).

### Dataset yang kita pakai
Sama dengan notebook U-Net: gabungan X-ray dada dari **Montgomery County** (138 citra) dan **Shenzhen No.3 People's Hospital** (566 citra), lengkap dengan mask paru (Jaeger dkk., 2014), diperkecil menjadi 128 × 128 piksel.

---
## Bagian 1 — Menyiapkan alat

In [ ]:
import time
import urllib.request
from pathlib import Path

import numpy as np                     # alat hitung untuk deretan angka
import matplotlib.pyplot as plt        # alat menggambar grafik
import matplotlib.patches as patches   # untuk menggambar kotak di atas citra
from scipy import ndimage              # untuk memisahkan "pulau-pulau" piksel pada mask
import torch                           # PyTorch: alat utama deep learning
from torchvision.transforms import v2        # alat augmentasi citra
from torchvision import tv_tensors           # "label jenis data": ini citra, ini mask
from torchvision.ops import masks_to_boxes   # menghitung kotak pembungkus dari mask

# Mask R-CNN siap pakai dari torchvision, beserta "kepala" yang akan kita ganti
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

# Angka acak dibuat "tetap" supaya hasilmu sama setiap kali notebook dijalankan ulang
torch.manual_seed(42)
np.random.seed(42)

print("PyTorch versi", torch.__version__)

### Memilih "mesin" untuk melatih model
Urutannya sama seperti biasa: **CUDA** (GPU NVIDIA, misalnya Colab) → **MPS** (GPU Mac Apple Silicon) → **CPU**.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")      # GPU NVIDIA
elif torch.backends.mps.is_available():
    device = torch.device("mps")       # GPU Apple Silicon
else:
    device = torch.device("cpu")       # prosesor biasa
    print("PERINGATAN: tanpa GPU, pelatihan Mask R-CNN akan sangat lambat!")

print("Model akan dilatih memakai:", device)

---
## Bagian 2 — Mengunduh dataset

Seluruh dataset tersimpan dalam satu file `paru_128.npz` (±10 MB) — file yang sama persis dengan notebook U-Net. Kalau file itu sudah ada di sebelah notebook, kita pakai langsung; kalau belum (misalnya di Colab), kita unduh dari GitHub.

In [ ]:
URL_REPO = "https://github.com/mctosima/ai-bm/raw/main/pertemuan-10-rcnn/"
file_data = Path("paru_128.npz")

if not file_data.exists():
    print("Mengunduh dataset ...")
    urllib.request.urlretrieve(URL_REPO + "paru_128.npz", file_data)

data = np.load(file_data)
citra_semua = data["citra"]      # citra X-ray, angka 0 (hitam) .. 255 (putih)
mask_semua = data["mask"]        # mask paru: 1 = paru, 0 = bukan paru
asal_semua = data["asal"]        # asal rumah sakit: "Montgomery" atau "Shenzhen"

print("Ukuran citra:", citra_semua.shape, "→ (jumlah citra, tinggi, lebar)")
print("Ukuran mask :", mask_semua.shape)
for rs in ["Montgomery", "Shenzhen"]:
    print(f"  {rs:10s}: {(asal_semua == rs).sum()} citra")

---
## Bagian 3 — Dari satu mask menjadi dua objek

Dataset kita hanya punya **satu mask per citra** (putih = paru), cocok untuk U-Net. Mask R-CNN butuh jawaban yang lebih rinci, yaitu daftar **objek**. Setiap objek terdiri dari:
1. **Kotak** (*bounding box*): `(x_kiri, y_atas, x_kanan, y_bawah)`.
2. **Label kelas**: `1` = paru kanan, `2` = paru kiri (`0` disisihkan untuk **latar**).
3. **Mask** objek itu sendiri.

Untungnya kedua paru di mask tampak sebagai dua **"pulau" putih yang terpisah** (dipisahkan jantung dan tulang belakang). Kita bisa memberi nomor setiap pulau secara otomatis dengan teknik **connected components**: piksel putih yang saling bersentuhan dianggap satu pulau.

Kadang ada serpihan kecil beberapa piksel akibat citra diperkecil. Serpihan itu kita abaikan: cukup ambil **2 pulau terbesar**. Pulau yang berada di sisi kiri citra adalah **paru kanan** pasien.

In [ ]:
NAMA_KELAS = {1: "paru kanan", 2: "paru kiri"}           # 0 = latar (bukan objek)
WARNA_KELAS = {1: (0.1, 0.8, 1.0), 2: (1.0, 0.6, 0.0)}    # biru muda & oranye


def hitung_kotak(masks):
    # Kotak terkecil yang membungkus setiap mask: (x_kiri, y_atas, x_kanan, y_bawah)
    kotak = masks_to_boxes(masks)
    kotak[:, 2:] += 1          # +1 supaya piksel paling kanan & paling bawah ikut masuk kotak
    return kotak


def mask_ke_objek(mask):
    pulau, jumlah_pulau = ndimage.label(mask)            # beri nomor 1, 2, 3, ... untuk setiap pulau
    luas = np.bincount(pulau.ravel())[1:]                 # luas (jumlah piksel) setiap pulau
    dua_terbesar = np.argsort(luas)[::-1][:2] + 1         # nomor 2 pulau terbesar; serpihan diabaikan

    daftar_mask, daftar_label = [], []
    for nomor in dua_terbesar:
        m = pulau == nomor
        kolom_tengah = np.where(m)[1].mean()              # posisi rata-rata pulau dari kiri ke kanan
        daftar_label.append(1 if kolom_tengah < mask.shape[1] / 2 else 2)
        daftar_mask.append(m)

    masks = torch.tensor(np.stack(daftar_mask), dtype=torch.uint8)    # bentuk (2, 128, 128)
    return {"boxes": hitung_kotak(masks), "labels": torch.tensor(daftar_label), "masks": masks}


target_semua = [mask_ke_objek(m) for m in mask_semua]

# Intip "jawaban" untuk citra #0
contoh_target = target_semua[0]
print("labels:", contoh_target["labels"].tolist(), "→", [NAMA_KELAS[int(k)] for k in contoh_target["labels"]])
print("boxes :\n", contoh_target["boxes"])
print("masks : bentuk", tuple(contoh_target["masks"].shape), "→ (jumlah objek, tinggi, lebar)")

Kita gambar supaya lebih jelas. Fungsi `tampilkan_objek` di bawah akan kita pakai berkali-kali: ia menggambar setiap objek dengan **mask berwarna, kotak, dan namanya** (plus skor keyakinan kalau objek itu tebakan model).

In [ ]:
def tampilkan_overlay(ax, citra, mask, warna=(0, 1, 0.4), judul=""):
    # Gambar citra hitam-putih, lalu tempelkan mask berwarna transparan di atasnya
    ax.imshow(citra, cmap="gray")
    lapisan = np.zeros((*mask.shape, 4))
    lapisan[mask == 1] = (*warna, 0.4)          # 0.4 = tingkat transparansi
    ax.imshow(lapisan)
    ax.set_title(judul)
    ax.axis("off")


def tampilkan_objek(ax, citra, objek, judul=""):
    # Gambar citra, lalu setiap objek: mask berwarna + kotak + nama kelas (+ skor kalau ada)
    ax.imshow(citra, cmap="gray")
    for k in range(len(objek["labels"])):
        kelas = int(objek["labels"][k])
        warna = WARNA_KELAS[kelas]
        lapisan = np.zeros((*citra.shape, 4))
        lapisan[objek["masks"][k].numpy() == 1] = (*warna, 0.45)
        ax.imshow(lapisan)
        x1, y1, x2, y2 = objek["boxes"][k].tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=warna, linewidth=1.5))
        teks = NAMA_KELAS[kelas]
        if "scores" in objek:
            teks += f" {objek['scores'][k]:.2f}"
        ax.text(x1 + 1, y1 + 7, teks, color="white", fontsize=7, bbox=dict(facecolor=warna, pad=1, linewidth=0))
    ax.set_title(judul)
    ax.axis("off")


contoh = [0, 300, 650]
fig, axes = plt.subplots(3, 3, figsize=(9, 9.5))
for baris, i in enumerate(contoh):
    axes[baris, 0].imshow(citra_semua[i], cmap="gray")
    axes[baris, 0].set_title(f"Citra #{i} ({asal_semua[i]})")
    axes[baris, 0].axis("off")
    tampilkan_overlay(axes[baris, 1], citra_semua[i], mask_semua[i], judul="Mask semantik (U-Net)")
    tampilkan_objek(axes[baris, 2], citra_semua[i], target_semua[i], judul="Objek (Mask R-CNN)")
plt.tight_layout()
plt.show()

Kolom tengah adalah jawaban yang dipelajari U-Net: satu warna untuk semua paru. Kolom kanan adalah jawaban yang dipelajari Mask R-CNN: **dua objek terpisah**, masing-masing dengan kotak dan namanya.

---
## Bagian 4 — Menyiapkan data dan cara menilai

### Data
Sama seperti notebook U-Net: angka 0–255 diubah menjadi 0–1, lalu data dibagi menjadi **latih**, **validasi**, dan **uji** dengan pengacakan yang **sama persis** (`manual_seed(0)`). Artinya data uji di sini = data uji U-Net, sehingga hasil kedua model bisa dibandingkan secara adil.

### Dice & IoU
Kita tetap memakai **Dice** dan **IoU** untuk menilai mask. Supaya bisa dibandingkan dengan U-Net, semua objek tebakan Mask R-CNN akan **digabung** dulu menjadi satu mask "paru vs bukan paru".
- **Dice** = 2 × luas irisan ÷ (luas tebakan + luas jawaban). 1 = sempurna, 0 = meleset total.
- **IoU** = luas irisan ÷ luas gabungan. Selalu sedikit lebih kecil dari Dice.

In [ ]:
# float() → angka desimal, /255 → skala 0..1, unsqueeze(1) → tambahkan dimensi kanal
X = torch.tensor(citra_semua).float().div(255).unsqueeze(1)
Y = torch.tensor(mask_semua).float().unsqueeze(1)          # mask semantik, untuk menghitung Dice
print("Bentuk X:", tuple(X.shape), " Bentuk Y:", tuple(Y.shape))

# Pembagian yang sama persis dengan notebook U-Net: 140 uji, 70 validasi, sisanya latih
urutan = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
indeks_uji, indeks_val, indeks_latih = urutan[:140], urutan[140:210], urutan[210:]
print(f"Latih: {len(indeks_latih)} | Validasi: {len(indeks_val)} | Uji: {len(indeks_uji)}")


def dice_batch(tebakan, jawaban):
    # Dice untuk setiap citra dalam satu batch
    irisan = (tebakan * jawaban).sum(dim=(1, 2, 3))
    return (2 * irisan + 1e-6) / (tebakan.sum(dim=(1, 2, 3)) + jawaban.sum(dim=(1, 2, 3)) + 1e-6)


def iou_batch(tebakan, jawaban):
    irisan = (tebakan * jawaban).sum(dim=(1, 2, 3))
    gabungan = tebakan.sum(dim=(1, 2, 3)) + jawaban.sum(dim=(1, 2, 3)) - irisan
    return (irisan + 1e-6) / (gabungan + 1e-6)

---
## Bagian 5 — Mengenal keluarga R-CNN

**R-CNN** = *Regions with CNN features*. Idenya: daripada memeriksa setiap posisi di citra, **usulkan dulu daerah-daerah yang mungkin berisi objek**, lalu periksa daerah itu dengan CNN. Keluarga ini berkembang dalam empat langkah:

| Model | Tahun | Apa yang baru? | Kecepatan |
|---|---|---|---|
| **R-CNN** | 2014 | ±2.000 kandidat kotak dari algoritma biasa (*selective search*); **setiap** potongan citra masuk CNN satu per satu | ±50 detik / citra |
| **Fast R-CNN** | 2015 | CNN dijalankan **sekali** untuk seluruh citra; kandidat kotak dipotong dari **peta fitur** (*RoI Pooling*) | ±2 detik / citra |
| **Faster R-CNN** | 2015 | Kandidat kotak dibuat oleh jaringan saraf itu sendiri: **Region Proposal Network (RPN)** | ±0,2 detik / citra |
| **Mask R-CNN** | 2017 | Tambah **kepala mask** di setiap kotak + *RoIAlign* (pemotongan fitur yang lebih presisi) | ±0,2 detik / citra |

### Bagaimana Mask R-CNN bekerja
```
citra ─► [Backbone: ResNet-50 + FPN] ─► peta fitur (CNN biasa, seperti encoder U-Net)
                                           │
                                           ├─► [RPN] ─► ±1.000 kandidat kotak: "mungkin ada objek di sini"
                                           │                     │
                                           └──► [RoIAlign] ◄─────┘  potong fitur di dalam setiap kotak
                                                     │
                       ┌─────────────────────────────┼─────────────────────────────┐
                       ▼                             ▼                             ▼
                [kepala kelas]                [kepala kotak]                [kepala mask]
          paru kanan / kiri / latar?    rapikan posisi kotaknya     mask 28×28 di dalam kotak
```
1. **Backbone** (*ResNet-50 + FPN*): CNN yang mengubah citra menjadi **peta fitur** — "ringkasan" citra, persis seperti encoder di U-Net. FPN (*Feature Pyramid Network*) menyiapkan peta fitur di beberapa ukuran supaya objek kecil maupun besar sama-sama terlihat.
2. **RPN** (*Region Proposal Network*): menebar ribuan kotak berbagai ukuran (*anchor*) dan menilai mana yang kemungkinan berisi objek.
3. **RoIAlign**: memotong peta fitur di dalam setiap kandidat kotak lalu menyeragamkan ukurannya.
4. **Tiga kepala** bekerja untuk setiap kotak: menebak **kelas**, **merapikan kotak**, dan **menggambar mask** kecil (28 × 28) yang kemudian direntangkan sesuai ukuran kotaknya.

Kepala mask itu pada dasarnya adalah **"U-Net mini" yang hanya bekerja di dalam satu kotak**.

### Transfer learning: tidak mulai dari nol
U-Net kita dilatih **dari nol** dan berhasil karena ukurannya kecil. Mask R-CNN punya ±44 juta parameter — terlalu banyak untuk dilatih dari nol dengan 494 citra.

Solusinya: **transfer learning**. Kita mengambil Mask R-CNN yang sudah dilatih pada **COCO** (±118 ribu foto sehari-hari dengan 80 jenis objek: orang, mobil, kucing, ...). Model ini sudah pandai mengenali tepi, tekstur, dan bentuk. Kita cukup:
1. Mengganti **kepala kelas** (91 kelas COCO → 3 kelas kita: latar, paru kanan, paru kiri).
2. Mengganti **kepala mask** (91 → 3).
3. Melatih ulang sebentar (*fine-tuning*) dengan citra X-ray kita.

Analoginya seperti merekrut fotografer berpengalaman lalu mengajarinya membaca X-ray — jauh lebih cepat daripada mengajari orang yang belum pernah melihat foto sama sekali.

In [ ]:
JUMLAH_KELAS = 3          # 0 = latar, 1 = paru kanan, 2 = paru kiri


def buat_model():
    # 1. Mask R-CNN yang sudah dilatih pada COCO (unduhan ±170 MB, hanya sekali).
    #    Bawaannya citra diperbesar ke 800 piksel; citra kita 128 piksel cukup diperbesar 2× ke 256.
    model = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT, min_size=256, max_size=256)

    # 2. Ganti kepala kelas & kotak: 91 kelas COCO → 3 kelas kita
    model.roi_heads.box_predictor = FastRCNNPredictor(1024, JUMLAH_KELAS)

    # 3. Ganti kepala mask: satu mask 28×28 untuk setiap kelas
    model.roi_heads.mask_predictor = MaskRCNNPredictor(256, 256, JUMLAH_KELAS)
    return model.to(device)


def ke_input(citra):
    # Mask R-CNN menerima DAFTAR citra 3 kanal (dibuat untuk foto berwarna).
    # X-ray kita hanya 1 kanal, jadi kanalnya disalin 3×: (1, 128, 128) → (3, 128, 128)
    return [c.expand(3, -1, -1).to(device) for c in citra]


torch.manual_seed(42)
model = buat_model()
print(f"Jumlah parameter: {sum(p.numel() for p in model.parameters()):,}")
print(f"Yang ikut dilatih: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Sebagian lapisan paling awal di backbone sengaja **dibekukan** (tidak ikut dilatih): lapisan-lapisan itu mengenali hal-hal umum seperti garis dan tepi, yang sama saja untuk foto kucing maupun X-ray.

Coba masukkan satu citra ke model yang **belum dilatih**. Keluarannya bukan satu gambar seperti U-Net, melainkan **daftar objek**:

In [ ]:
model.eval()                                   # mode ujian
with torch.no_grad():
    keluaran = model(ke_input(X[:1]))[0]       # [0] = hasil untuk citra pertama

for kunci, nilai in keluaran.items():
    print(f"{kunci:6s}: bentuk {tuple(nilai.shape)}")
print("\nN = jumlah objek yang ditemukan. Kepala baru masih acak, jadi tebakannya belum berarti apa-apa.")

- `boxes` (N, 4): kotak setiap objek.
- `labels` (N,): kelas setiap objek (1 atau 2).
- `scores` (N,): **skor keyakinan** 0–1 untuk setiap objek.
- `masks` (N, 1, 128, 128): **peluang** setiap piksel milik objek itu — mirip keluaran U-Net setelah `sigmoid`, tetapi satu untuk **setiap objek**.

---
## Bagian 6 — Melatih model

### Lima loss sekaligus
Pada mode latih (`model.train()`), Mask R-CNN menerima citra **beserta jawabannya** dan langsung mengembalikan **lima loss** — satu untuk setiap bagian model:

| Loss | Bagian | Menilai apa? |
|---|---|---|
| `loss_objectness` | RPN | Kandidat kotak ini berisi objek atau latar? |
| `loss_rpn_box_reg` | RPN | Seberapa meleset posisi kandidat kotak? |
| `loss_classifier` | Kepala kelas | Paru kanan, paru kiri, atau latar? |
| `loss_box_reg` | Kepala kotak | Seberapa meleset kotak akhirnya? |
| `loss_mask` | Kepala mask | BCE per piksel di dalam kotak — sama dengan BCE di U-Net |

Kelima loss itu cukup **dijumlahkan**, lalu resep 5 langkah berjalan seperti biasa. Bedanya: langkah 1 (tebak) dan 2 (hitung loss) dilakukan model dalam satu panggilan.

### Fungsi-fungsi pembantu
- `tebak()`: menjalankan model pada sekumpulan citra dan mengumpulkan objek-objeknya.
- `saring()`: membuang objek yang **skornya** di bawah batas (bawaan 0,5).
- `gabung_objek()`: menyatukan semua objek menjadi satu mask "paru vs bukan paru" untuk menghitung Dice.
- `latih()`: resep latihan, bisa menerima `augmentasi` (dipakai di Bagian 9).

In [ ]:
def tebak(model, indeks, ukuran_batch=16):
    # Daftar objek tebakan untuk setiap citra (sudah dipindah ke CPU)
    model.eval()
    hasil = []
    with torch.no_grad():
        for awal in range(0, len(indeks), ukuran_batch):
            for k in model(ke_input(X[indeks[awal : awal + ukuran_batch]])):
                hasil.append({
                    "boxes": k["boxes"].cpu(),
                    "labels": k["labels"].cpu(),
                    "scores": k["scores"].cpu(),
                    "masks": (k["masks"][:, 0] > 0.5).to(torch.uint8).cpu(),   # peluang piksel → 0/1
                })
    return hasil


def saring(daftar_objek, batas_skor=0.5):
    # Simpan hanya objek yang skor keyakinannya di atas batas
    return [{kunci: nilai[o["scores"] > batas_skor] for kunci, nilai in o.items()} for o in daftar_objek]


def gabung_objek(daftar_objek):
    # Semua objek dalam satu citra digabung menjadi satu mask semantik (bentuk: jumlah citra, 1, 128, 128)
    kosong = torch.zeros(X.shape[-2:], dtype=torch.uint8)
    return torch.stack([o["masks"].amax(dim=0) if len(o["masks"]) else kosong for o in daftar_objek]).float().unsqueeze(1)


def evaluasi(model, indeks):
    # Dice rata-rata pada sekumpulan citra (mode ujian, tanpa belajar)
    tebakan = gabung_objek(saring(tebak(model, indeks)))
    return dice_batch(tebakan, Y[indeks]).mean().item()


def latih(model, indeks, jumlah_epoch, augmentasi=None, indeks_val=None, ukuran_batch=8):
    parameter = [p for p in model.parameters() if p.requires_grad]     # hanya yang tidak dibekukan
    optimizer = torch.optim.AdamW(parameter, lr=1e-4)
    riwayat = []
    for epoch in range(jumlah_epoch):
        model.train()                                         # mode belajar
        acak = indeks[torch.randperm(len(indeks))]            # acak urutan setiap epoch
        total = {}

        for awal in range(0, len(acak), ukuran_batch):
            batch = acak[awal : awal + ukuran_batch]
            citra, target = X[batch], [target_semua[i] for i in batch]

            if augmentasi is not None:                        # dipakai di Bagian 9
                citra, target = terapkan_augmentasi(augmentasi, citra, target)

            target = [{kunci: nilai.to(device) for kunci, nilai in t.items()} for t in target]
            semua_loss = model(ke_input(citra), target)      # 1+2. tebak & hitung kelima loss
            loss = sum(semua_loss.values())                   #      lalu jumlahkan
            optimizer.zero_grad()                             # 3. bersihkan koreksi lama
            loss.backward()                                   # 4. cari arah perbaikan
            optimizer.step()                                  # 5. perbaiki model
            for nama, nilai in semua_loss.items():
                total[nama] = total.get(nama, 0.0) + nilai.item() * len(batch)

        loss_rata = {nama: nilai / len(indeks) for nama, nilai in total.items()}
        if indeks_val is not None:
            dice_val = evaluasi(model, indeks_val)
            riwayat.append((loss_rata, dice_val))
            print(f"Epoch {epoch + 1:2d}/{jumlah_epoch} | loss total = {sum(loss_rata.values()):.4f} "
                  f"| loss_mask = {loss_rata['loss_mask']:.4f} | Dice validasi = {dice_val:.3f}")
    return riwayat

### Saatnya melatih!
Satu epoch Mask R-CNN jauh lebih berat daripada U-Net. Di GPU T4 Colab, JUMLAH_EPOCH epoch memakan waktu beberapa menit.

Optimizer yang kita pakai adalah **AdamW** dengan *learning rate* kecil (0,0001). Model *pretrained* sudah "pintar"; langkah perbaikan yang terlalu besar justru bisa merusak pengetahuannya — bahkan membuat loss meledak menjadi `nan`.

In [ ]:
JUMLAH_EPOCH = 8

torch.manual_seed(42)
model = buat_model()

mulai = time.time()
riwayat = latih(model, indeks_latih, JUMLAH_EPOCH, indeks_val=indeks_val)
print(f"Selesai dalam {time.time() - mulai:.0f} detik.")
torch.save(model.state_dict(), "model_maskrcnn_paru.pt")     # simpan hasil belajar (±170 MB)

In [ ]:
epoch_ke = range(1, JUMLAH_EPOCH + 1)
fig, (kiri, kanan) = plt.subplots(1, 2, figsize=(12, 3.8))
for nama in riwayat[0][0]:
    kiri.plot(epoch_ke, [r[0][nama] for r in riwayat], marker="o", label=nama)
kiri.set_title("Kelima loss latih (makin kecil makin baik)")
kiri.set_xlabel("epoch")
kiri.legend(fontsize=8)
kanan.plot(epoch_ke, [r[1] for r in riwayat], marker="o", color="tab:green")
kanan.set_title("Dice validasi (makin besar makin baik)")
kanan.set_xlabel("epoch")
plt.show()

Perhatikan bahwa `loss_mask` biasanya menjadi loss terbesar: menggambar mask yang tepat memang bagian tersulit. Sementara itu, mencari *letak* paru (kotak) relatif mudah — paru selalu berada di tempat yang kurang lebih sama.

---
## Bagian 7 — Ujian: menilai model dengan data uji

In [ ]:
objek_uji_mentah = tebak(model, indeks_uji)       # semua objek, termasuk yang skornya rendah
objek_uji = saring(objek_uji_mentah)               # hanya yang skornya > 0,5
tebakan_uji = gabung_objek(objek_uji)
jawaban_uji = Y[indeks_uji]

dice_per_citra = dice_batch(tebakan_uji, jawaban_uji)
iou_per_citra = iou_batch(tebakan_uji, jawaban_uji)
jumlah_objek = np.array([len(o["labels"]) for o in objek_uji])

print(f"Dice rata-rata data uji: {dice_per_citra.mean():.3f}")
print(f"IoU  rata-rata data uji: {iou_per_citra.mean():.3f}")
print(f"Jumlah objek per citra : {dict(zip(*np.unique(jumlah_objek, return_counts=True)))}  → {{jumlah objek: banyak citra}}")
print("(Bandingkan dengan U-Net di notebook sebelumnya: Dice 0,954 dan IoU 0,914)")

### Melihat kesalahannya
Kita tampilkan 2 citra dengan Dice **terbaik** dan 2 citra dengan Dice **terburuk**. Pada kolom terakhir:
- **Hijau** = benar paru.
- **Merah** = *false positive*: model bilang paru, padahal bukan.
- **Biru** = *false negative*: paru yang **terlewat** oleh model.

In [ ]:
def peta_kesalahan(tebakan, jawaban):
    gambar = np.zeros((*jawaban.shape, 3))
    gambar[(tebakan == 1) & (jawaban == 1)] = (0.1, 0.8, 0.3)    # hijau: benar
    gambar[(tebakan == 1) & (jawaban == 0)] = (0.9, 0.2, 0.2)    # merah: false positive
    gambar[(tebakan == 0) & (jawaban == 1)] = (0.2, 0.4, 1.0)    # biru: false negative
    return gambar


urut = dice_per_citra.argsort()                     # dari Dice terkecil ke terbesar
dipilih = [urut[-1], urut[-2], urut[1], urut[0]]    # 2 terbaik, 2 terburuk

fig, axes = plt.subplots(4, 4, figsize=(12, 12.5))
for baris, j in enumerate(dipilih):
    i = int(indeks_uji[j])
    axes[baris, 0].imshow(citra_semua[i], cmap="gray")
    axes[baris, 0].set_title(f"Citra ({asal_semua[i]})")
    tampilkan_objek(axes[baris, 1], citra_semua[i], target_semua[i], judul="Jawaban benar")
    tampilkan_objek(axes[baris, 2], citra_semua[i], objek_uji[j], judul="Tebakan model")
    axes[baris, 3].imshow(peta_kesalahan(tebakan_uji[j, 0].numpy(), jawaban_uji[j, 0].numpy()))
    axes[baris, 3].set_title(f"Dice = {dice_per_citra[j]:.3f}")
    for ax in axes[baris]:
        ax.axis("off")
plt.tight_layout()
plt.show()

### Bonus segmentasi instance: luas setiap paru
Karena setiap paru adalah objek tersendiri, kita bisa langsung mengukur **luas paru kanan dan paru kiri** secara terpisah — sesuatu yang tidak bisa dilakukan U-Net tanpa langkah tambahan. Untuk setiap kelas kita ambil objek dengan skor tertinggi.

In [ ]:
def luas_per_kelas(objek):
    # Luas (piksel) objek dengan skor tertinggi untuk setiap kelas; 0 kalau kelas itu tidak ditemukan
    luas = {}
    for kelas in NAMA_KELAS:
        cocok = objek["labels"] == kelas
        if "scores" in objek and cocok.any():
            terbaik = objek["scores"].masked_fill(~cocok, -1).argmax()
            luas[kelas] = int(objek["masks"][terbaik].sum())
        else:
            luas[kelas] = int(objek["masks"][cocok].sum())
    return luas


print("              |  paru kanan (px)  |  paru kiri (px)  | rasio kanan:kiri")
print("              | jawaban   tebakan | jawaban  tebakan | jawaban  tebakan")
for j in range(8):
    benar = luas_per_kelas(target_semua[int(indeks_uji[j])])
    tebakan = luas_per_kelas(objek_uji[j])
    rasio_benar = benar[1] / max(benar[2], 1)
    rasio_tebakan = tebakan[1] / max(tebakan[2], 1)
    print(f"Citra uji #{j:<2d} | {benar[1]:7d}  {tebakan[1]:7d}  | {benar[2]:7d}  {tebakan[2]:7d} "
          f"|  {rasio_benar:5.2f}   {rasio_tebakan:5.2f}")

Paru kanan normalnya sedikit **lebih besar** dari paru kiri (paru kiri berbagi tempat dengan jantung), sehingga rasionya biasanya di atas 1. Rasio yang jauh menyimpang bisa menjadi tanda ada yang perlu diperiksa radiolog.

### Batas skor itu bukan harga mati
Di U-Net kita mengatur batas **peluang piksel**. Di Mask R-CNN ada satu tombol lagi: batas **skor objek**. Objek dengan skor di bawah batas dibuang. Batas terlalu rendah → objek "palsu" ikut masuk; batas terlalu tinggi → paru yang sebenarnya bisa ikut terbuang.

In [ ]:
for batas in [0.05, 0.3, 0.5, 0.7, 0.9, 0.99]:
    tersaring = saring(objek_uji_mentah, batas)
    dice = dice_batch(gabung_objek(tersaring), jawaban_uji).mean()
    rata_objek = np.mean([len(o["labels"]) for o in tersaring])
    print(f"Batas skor {batas:.2f} → Dice = {dice:.3f} | rata-rata objek per citra = {rata_objek:.2f}")

---
## Bagian 8 — Augmentasi untuk deteksi

Aturan emas dari notebook U-Net masih berlaku: **citra dan mask harus berubah bersamaan**. Di Mask R-CNN ada satu hal lagi yang wajib ikut berubah: **kotak**.

Cara paling aman: augmentasi citra **dan semua mask objek** dengan `tv_tensors.Image` dan `tv_tensors.Mask`, lalu **hitung ulang kotak dari mask** hasil augmentasi. (Memutar kotaknya langsung akan menghasilkan kotak yang terlalu longgar, karena sudut-sudut kotak yang diputar ikut melebar.)

### Augmentasi harus masuk akal secara klinis
Kita tetap **tidak** memakai *flip* kiri-kanan. Di notebook U-Net alasannya karena jantung akan pindah ke kanan (*dextrocardia*). Di sini ada alasan kedua: kalau citra dicerminkan, paru kanan pindah ke sisi kanan citra — **label "paru kanan" dan "paru kiri" menjadi tertukar**!

In [ ]:
aug_geometri = v2.RandomAffine(degrees=10, translate=(0.1, 0.1), scale=(0.85, 1.15))  # putar, geser, perbesar
aug_intensitas = v2.ColorJitter(brightness=0.5, contrast=0.5)                         # terang/gelap, kontras
aug_lengkap = v2.Compose([aug_geometri, aug_intensitas])


def terapkan_augmentasi(augmentasi, citra, target):
    # Augmentasi setiap pasangan (citra, objek) dalam batch; acakannya berbeda untuk setiap pasangan
    hasil_citra, hasil_target = [], []
    for c, t in zip(citra, target):
        c_baru, m_baru = augmentasi(tv_tensors.Image(c), tv_tensors.Mask(t["masks"]))
        masih_ada = m_baru.flatten(1).sum(dim=1) > 0      # buang objek yang terdorong keluar citra
        m_baru = m_baru[masih_ada]
        hasil_citra.append(c_baru)
        hasil_target.append({"boxes": hitung_kotak(m_baru), "labels": t["labels"][masih_ada], "masks": m_baru})
    return torch.stack(hasil_citra), hasil_target


# Lihat 5 variasi acak dari citra yang sama
i = 0
fig, axes = plt.subplots(1, 6, figsize=(17, 3.2))
tampilkan_objek(axes[0], X[i, 0].numpy(), target_semua[i], judul="Asli")
for ax in axes[1:]:
    c, t = terapkan_augmentasi(aug_lengkap, X[i : i + 1], [target_semua[i]])
    tampilkan_objek(ax, c[0, 0].numpy(), t[0], judul="Augmentasi")
plt.tight_layout()
plt.show()

Mask **dan** kotak selalu ikut bergerak bersama paru, dan labelnya tetap benar.

---
## Bagian 9 — Eksperimen: model dari rumah sakit lain

### Skenario
Sama dengan notebook U-Net: rumah sakit di **Shenzhen** punya model yang dilatih dengan **hanya 20 citra** miliknya, lalu dipakai di rumah sakit **Montgomery** — mesin X-ray berbeda, pengaturan berbeda, pasien berbeda.

Pertanyaan tambahan kali ini: **apakah transfer learning dari COCO membuat Mask R-CNN lebih tahan terhadap perbedaan rumah sakit?**

Kita melatih model baru dengan 20 citra Shenzhen dalam tiga skenario — tanpa augmentasi, augmentasi geometri, dan augmentasi intensitas — masing-masing diulang **3 kali** dengan 20 citra yang berbeda, lalu diambil rata-ratanya.

> Eksperimen ini melatih 9 model Mask R-CNN, jadi butuh waktu lebih lama (di GPU T4 sekitar 10 menit). Kalau waktu mepet, ubah `JUMLAH_ULANGAN = 1`.

In [ ]:
indeks_shenzhen = torch.tensor(np.where(asal_semua == "Shenzhen")[0])
indeks_montgomery = torch.tensor(np.where(asal_semua == "Montgomery")[0])

# 20 citra Shenzhen dari data latih (tidak ada yang berasal dari data uji)
shenzhen_latih = indeks_latih[torch.from_numpy(asal_semua[indeks_latih.numpy()] == "Shenzhen")]
shenzhen_uji = indeks_uji[torch.from_numpy(asal_semua[indeks_uji.numpy()] == "Shenzhen")]

skenario = {
    "Tanpa augmentasi": None,
    "Geometri": aug_geometri,
    "Intensitas": aug_intensitas,
}
JUMLAH_ULANGAN = 3
EPOCH_KECIL = 25
hasil = {}
mulai = time.time()
for nama, augmentasi in skenario.items():
    catatan = []
    for ulangan in range(JUMLAH_ULANGAN):
        # Pilih 20 citra Shenzhen secara acak; ulangan ke-n memakai 20 citra yang sama di semua skenario
        pilih = torch.randperm(len(shenzhen_latih), generator=torch.Generator().manual_seed(ulangan))[:20]
        torch.manual_seed(ulangan)                       # titik awal kepala baru juga sama antar-skenario
        model_kecil = buat_model()
        latih(model_kecil, shenzhen_latih[pilih], jumlah_epoch=EPOCH_KECIL, augmentasi=augmentasi)
        catatan.append((evaluasi(model_kecil, shenzhen_uji), evaluasi(model_kecil, indeks_montgomery)))
    hasil[nama] = np.array(catatan)                      # bentuk (ulangan, 2 rumah sakit)
    print(f"{nama:17s} | Dice Montgomery tiap ulangan = {np.round(hasil[nama][:, 1], 3)} "
          f"| rata-rata Shenzhen = {hasil[nama][:, 0].mean():.3f}, Montgomery = {hasil[nama][:, 1].mean():.3f}")
print(f"Total waktu: {time.time() - mulai:.0f} detik")

In [ ]:
nama_skenario = list(hasil)
posisi = np.arange(len(nama_skenario))
plt.figure(figsize=(8, 4))
plt.bar(posisi - 0.2, [hasil[n][:, 0].mean() for n in nama_skenario], width=0.4, label="Uji: Shenzhen (RS sama)", color="tab:green")
plt.bar(posisi + 0.2, [hasil[n][:, 1].mean() for n in nama_skenario], width=0.4, label="Uji: Montgomery (RS lain)", color="tab:orange")
plt.xticks(posisi, nama_skenario)
plt.ylabel("Dice")
plt.ylim(0, 1)
plt.title(f"Mask R-CNN, dilatih dengan 20 citra Shenzhen (rata-rata {JUMLAH_ULANGAN} ulangan)")
plt.legend(loc="lower right")
plt.show()

### Apa yang kita pelajari?
@@KESIMPULAN@@

---
## Rangkuman: U-Net vs Mask R-CNN

| | U-Net | Mask R-CNN |
|---|---|---|
| Jenis tugas | Segmentasi **semantik** | Segmentasi **instance** |
| Keluaran | 1 mask untuk seluruh citra | Untuk setiap objek: kotak + kelas + skor + mask |
| Cara kerja | Encoder–decoder, setiap piksel dinilai langsung | Cari kandidat kotak (RPN) → kelas, kotak, dan mask per kotak |
| Ukuran model | ±0,5 juta parameter | ±44 juta parameter |
| Cara melatih | Dari nol | Transfer learning dari COCO |
| Waktu latih | Detik | Menit (butuh GPU) |
| Dice data uji | 0,954 | @@DICE@@ |
| Tepi mask | Tajam (mask seukuran citra) | Sedikit lebih kasar (mask 28 × 28 direntangkan ke ukuran kotak) |
| Bisa memisahkan paru kanan & kiri? | Tidak (perlu langkah tambahan) | **Ya**, langsung |

**Kapan memakai yang mana?**
- Cukup tahu *piksel mana* yang termasuk suatu jaringan (paru, tumor, pembuluh darah)? → **U-Net**: kecil, cepat, tepinya tajam.
- Perlu **menghitung dan memisahkan** objek satu per satu (sel, nodul, gigi, kedua paru)? → **Mask R-CNN**.

### Untuk dicoba sendiri
1. Ubah menjadi **satu kelas** saja ("paru") dengan `JUMLAH_KELAS = 2` dan semua label diganti `1`. Apakah Dice berubah?
2. Ubah `min_size=256, max_size=256` menjadi `128` (lebih cepat) atau `512` (lebih detail). Bagaimana pengaruhnya pada Dice dan waktu latih?
3. Tambahkan `v2.RandomHorizontalFlip()` ke augmentasi lalu lihat hasil di Bagian 8. Apa yang terjadi dengan label paru kanan/kiri?
4. Latih **tanpa** transfer learning: `maskrcnn_resnet50_fpn(weights=None, weights_backbone=None, ...)`. Seberapa jauh Dice turun?
5. Ganti dengan **Faster R-CNN** (`fasterrcnn_resnet50_fpn`) yang hanya menghasilkan kotak, tanpa mask.

### Referensi
- Girshick, R., Donahue, J., Darrell, T., & Malik, J. (2014). *Rich feature hierarchies for accurate object detection and semantic segmentation* (R-CNN). CVPR. [arXiv:1311.2524](https://arxiv.org/abs/1311.2524)
- Girshick, R. (2015). *Fast R-CNN*. ICCV. [arXiv:1504.08083](https://arxiv.org/abs/1504.08083)
- Ren, S., He, K., Girshick, R., & Sun, J. (2015). *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks*. NeurIPS. [arXiv:1506.01497](https://arxiv.org/abs/1506.01497)
- He, K., Gkioxari, G., Dollár, P., & Girshick, R. (2017). *Mask R-CNN*. ICCV. [arXiv:1703.06870](https://arxiv.org/abs/1703.06870)
- Lin, T.-Y., dkk. (2017). *Feature Pyramid Networks for Object Detection*. CVPR. [arXiv:1612.03144](https://arxiv.org/abs/1612.03144)
- Lin, T.-Y., dkk. (2014). *Microsoft COCO: Common Objects in Context*. ECCV. [arXiv:1405.0312](https://arxiv.org/abs/1405.0312)
- Ronneberger, O., Fischer, P., & Brox, T. (2015). *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI. [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)
- Jaeger, S., dkk. (2014). *Two public chest X-ray datasets for computer-aided screening of pulmonary diseases*. Quantitative Imaging in Medicine and Surgery, 4(6), 475–477.
- Dataset Kaggle: [Chest X-ray Dataset for Tuberculosis Segmentation](https://www.kaggle.com/datasets/iamtapendu/chest-x-ray-lungs-segmentation) (Apache 2.0).